# Week 3 — development evaluation
No paid APIs. GPU-dependent code remains in this notebook. Run installation, restart session, then run cells in order. Based on the successfully executed Week 2 pipeline; Week 3 GPU behavior is not yet verified.

This is DEVELOPMENT work. Reserved final questions are never evaluated here. Prior exposure to those questions is uncertain; no clean held-out claim is authorized by this split.

Use a new run name for every changed setting. Different notebooks/conditions must not write the same Drive folder. The exported ZIP is needed to finish the experimental portion of Week 3.

Reader format revision: explicit instructions are separated from question text; control gates stop evaluation when basic answer formatting fails. Sampling remains disabled. Use the new default run name so cached old answers are not reused.


## 1. Install; then restart the session

In [ ]:
import subprocess, sys
packages = [
    'huggingface-hub==0.36.0', 'transformers==4.51.3', 'accelerate==1.7.0', 'bitsandbytes==0.50.2',
    'sentence-transformers==4.1.0', 'datasets==3.6.0',
    'qwen-agent==0.0.29', 'openai==1.78.1', 'python-dotenv==1.1.0',
    'json_repair==0.51.0', 'rank-bm25==0.2.2', 'tiktoken==0.9.0',
    'nltk==3.9.1', 'rapidfuzz==3.13.0', 'pyyaml==6.0.2', 'scikit-learn>=1.5,<2'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Installation complete. Restart the session now, then continue at step 2. Do not rerun the old install cell.')
print('The OpenAI library is only an upstream import dependency; API clients are blocked below.')

# Test the installed files in a fresh process; the notebook kernel still needs a restart.
subprocess.run([sys.executable, '-c', 'import huggingface_hub; from huggingface_hub import HfApi; import huggingface_hub._commit_api; print("Hub import check:", huggingface_hub.__version__)'], check=True)


## 2. Configuration
For the Qwen writer baseline, change WRITER_ID to `Qwen/Qwen3-4B` and use a new run name. For the extension, choose Mem-alpha and CONDITION=`retrieval`. Reader settings are provisional: select them in reader diagnostics before matched development runs.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
RUN_NAME = 'week3_development_memalpha_baseline_reader_v2_01'
OUTPUT = Path('/content/drive/MyDrive/CLMM_week3') / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
WRITER_ID = 'YuWangX/Memalpha-4B'
READER_ID = 'Qwen/Qwen3-4B'
EMBED_ID = 'sentence-transformers/all-MiniLM-L6-v2'
CHUNK_TOKENS = 256  # Week 1 used 1024; see header
WRITE_TOKENS = 2048
ANSWER_TOKENS = 512
MAX_INPUT_PLUS_OUTPUT = 16384  # Week 1 used 8192; MH memory exceeded it once facts stopped being lost
QUESTIONS_PER_CONTEXT = 20
SEED = 42
MAX_STAGE_MINUTES = 90  # Checked between generations; cannot interrupt an in-flight generation.
MEMALPHA_COMMIT = '9c0e1f6482f8ea7a322012a8155247966285946f'
MAB_COMMIT = '538026089d1a8a8eff05121d0db89b388f360eba'
print('Results:', OUTPUT)

TASK = 'development_evaluation'
CONDITION = 'baseline'  # baseline, prompted, retrieval; development runs use baseline or retrieval
READER_STYLE = 'explicit'  # original or explicit; compare on development, then freeze
READER_THINKING = False
SPLIT = {'dataset_revision': '7ea066982b140a19337e17e60d45d4076e042faf', 'history_sha256': 'e46cb14b53eedd71bf92762974a597d46b001157b6b769c03865dee9641818a1', 'seed': 42, 'exposure_status': 'uncertain: team cannot confirm which additional questions were inspected', 'claim': 'reserved questions from a previously inspected history; NOT certified untouched', 'tasks': {'factconsolidation_sh_6k': {'development': [0, 1, 2, 3, 4, 8, 9, 16, 18, 19, 22, 33, 36, 40, 59, 74, 80, 86, 91, 99], 'reserved_final': [5, 6, 7, 10, 11, 12, 13, 14, 15, 17, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 37, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 81, 82, 83, 84, 85, 87, 88, 89, 90, 92, 93, 94, 95, 96, 97, 98]}, 'factconsolidation_mh_6k': {'development': [0, 1, 2, 3, 4, 8, 16, 30, 32, 33, 34, 58, 62, 69, 74, 76, 82, 88, 94, 96], 'reserved_final': [5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 60, 61, 63, 64, 65, 66, 67, 68, 70, 71, 72, 73, 75, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 89, 90, 91, 92, 93, 95, 97, 98, 99]}}}
PINNED_REVISIONS = {'Qwen/Qwen3-4B': '1cfa9a7208912126459214e8b04321603b3df60c', 'sentence-transformers/all-MiniLM-L6-v2': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41', 'YuWangX/Memalpha-4B': '8c5683d0196a846ef1048b1fd0d4f9c7196e22d6', 'dataset': '7ea066982b140a19337e17e60d45d4076e042faf'}
PROTOCOL_SOURCE = '"""CPU-only Week 3 protocol helpers. No benchmark answers enter writer decisions."""\nimport json\nimport re\nfrom collections import Counter\n\nPROVENANCE = \'\'\'Preserve the original source number on EVERY stored fact line as [source_fact_id=N].\nUse only numbers present in the supplied source or existing memory; never invent or renumber them.\nCopying or rewriting a fact does not change its source number. A note with several facts needs a number on each fact line.\nOperation timestamps and chunk indices are NOT freshness evidence. Preserve these annotations for the answering model.\'\'\'\nCONFLICT = \'\'\'Before writing, check whether incoming facts conflict with existing memory.\nFor the SAME subject and relation, a larger source fact number supersedes a smaller one.\nUpdate the existing entry to the newer fact when appropriate; identical facts need no duplicate.\nUnrelated facts and legitimately coexisting facts must remain. If a note contains several facts,\nedit only the conflicting part and preserve all other lines and their original numbers.\nUse the actual memory ID for update/delete tools. Do not delete a whole multi-fact note to remove one fact.\'\'\'\n\ndef strict_calls(messages):\n    accepted, rejected = [], []\n    for message in messages:\n        call = message.get(\'function_call\')\n        if not call:\n            continue\n        try:\n            args = call[\'arguments\']\n            if isinstance(args, str):\n                args = json.loads(args)\n            if not isinstance(args, dict):\n                raise ValueError(\'Arguments must be a JSON object\')\n            if call[\'name\'] not in {\'new_memory_insert\', \'memory_update\', \'memory_delete\', \'search_memory\'}:\n                raise ValueError(\'Unknown tool\')\n            accepted.append({\'name\': call[\'name\'], \'arguments\': args})\n        except (ValueError, KeyError, TypeError) as error:\n            rejected.append({\'raw_call\': call, \'reason\': str(error)})\n    return accepted, rejected\n\ndef tokens(text):\n    return set(re.findall(r\'[a-z]+\', text.lower())) - {\'the\',\'is\',\'in\',\'of\',\'a\',\'an\',\'and\',\'with\',\'to\',\'source\',\'fact\',\'id\'}\n\ndef retrieve_entries(state, chunk, k=6):\n    """Lexical overlap, ranked per incoming line then globally; includes whole notes.\n    No gold conflict parser, answers, or future facts are used. Zero-overlap entries are excluded.\n    """\n    queries = [tokens(line) for line in chunk.splitlines() if line.strip()]\n    candidates = []\n    for kind in [\'semantic\', \'episodic\']:\n        for entry in state[kind]:\n            for mid, content in entry.items():\n                words = tokens(content)\n                score = max((len(q & words) / max(1, len(q)) for q in queries), default=0)\n                if score:\n                    candidates.append({\'memory_type\': kind+\'_memory\', \'memory_id\': mid,\n                                       \'content\': content, \'retrieval_score\': score})\n    return sorted(candidates, key=lambda e: (-e[\'retrieval_score\'], e[\'memory_type\'], e[\'memory_id\']))[:k]\n\ndef writer_suffix(state, chunk, condition):\n    if condition not in {\'baseline\',\'prompted\',\'retrieval\'}:\n        raise ValueError(condition)\n    candidates = retrieve_entries(state, chunk) if condition == \'retrieval\' else []\n    suffix = \'\\n\\n\' + PROVENANCE\n    if condition != \'baseline\':\n        suffix += \'\\n\\n\' + CONFLICT\n    if candidates:\n        suffix += \'\\n\\nExisting candidate notes (not new facts):\\n\' + json.dumps(candidates, ensure_ascii=False)\n    return suffix, candidates\n\ndef numbered_lines(state):\n    """Diagnostics only: counts IDs and malformed/missing fact-line markers, not semantic correctness."""\n    lines = [line for e in state[\'semantic\'] for text in e.values() for line in text.splitlines() if line.strip()]\n    annotated = [line for line in lines if re.match(r\'^\\s*\\[source_fact_id=\\d+\\]\', line)]\n    return {\'semantic_nonempty_lines\': len(lines), \'annotated_lines\': len(annotated),\n            \'missing_marker_lines\': len(lines)-len(annotated)}\n\ndef diagnostic_cases():\n    old = \'[source_fact_id=1] Ada lives in Paris.\'\n    unrelated = \'[source_fact_id=2] Ben likes chess.\'\n    fillers = [f\'[source_fact_id={i+3}] Person{i} likes gardening.\' for i in range(6)]\n    return [\n        {\'id\':\'visible\',\'seed\':[old, unrelated], \'incoming\':\'20. Ada now lives in Rome.\', \'expected_new\':\'Rome\', \'expected_old\':\'Paris\', \'retain\':\'Ben likes chess\'},\n        {\'id\':\'hidden\',\'seed\':[old, unrelated]+fillers, \'incoming\':\'20. Ada now lives in Rome.\', \'expected_new\':\'Rome\', \'expected_old\':\'Paris\', \'retain\':\'Ben likes chess\'},\n        {\'id\':\'block\',\'seed\':[old+\'\\n\'+unrelated], \'incoming\':\'20. Ada now lives in Rome.\', \'expected_new\':\'Rome\', \'expected_old\':\'Paris\', \'retain\':\'Ben likes chess\'},\n        {\'id\':\'duplicate\',\'seed\':[old,unrelated], \'incoming\':\'1. Ada lives in Paris.\', \'expected_new\':\'Paris\',\'expected_old\':None,\'retain\':\'Ben likes chess\'},\n        {\'id\':\'coexist\',\'seed\':[\'[source_fact_id=1] Ada likes painting.\',unrelated], \'incoming\':\'20. Ada also likes hiking.\', \'expected_new\':\'hiking\',\'expected_old\':None,\'retain\':\'Ada likes painting\'},\n        {\'id\':\'older_arrives_late\',\'seed\':[\'[source_fact_id=20] Ada lives in Rome.\',unrelated], \'incoming\':\'1. Ada lived in Paris at source time 1.\', \'expected_new\':\'Rome\',\'expected_old\':None,\'retain\':\'Ben likes chess\'},\n    ]\n\ndef reader_messages(knowledge, question, style, original_template):\n    """Keep task instructions out of the question field; never translate IDs into gold answers."""\n    if style == \'original\':\n        return [{\'role\':\'system\',\'content\':\'Answer using only the supplied knowledge. If it does not contain the answer, say unknown.\'},\n                {\'role\':\'user\',\'content\':\'[Knowledge Pool]\\n\'+knowledge+\'\\n\\n\'+original_template.format(question=question)}]\n    if style != \'explicit\':\n        raise ValueError(\'Unknown reader style: \'+style)\n    return [\n        {\'role\':\'system\',\'content\':(\n            \'Answer the question using only the supplied facts, even if they contradict real-world knowledge. \'\n            \'A number at the start of a fact, or source_fact_id, is ordering metadata, NOT its answer. \'\n            \'Only when facts about the same subject and relation conflict, use the fact with the largest source number. \'\n            \'Keep unrelated facts. Follow multiple relations when the question requires it. \'\n            \'Output ONLY the requested factual value: a name, place, activity, or other answer. \'\n            \'Do not output a source fact number, copy a fact line, explain your reasoning, or add an Answer label. \'\n            \'If the facts do not supply the answer, output unknown.\')},\n        {\'role\':\'user\',\'content\':\'<facts>\\n\'+knowledge+\'\\n</facts>\\n\\n<question>\\n\'+question+\'\\n</question>\'}]\n\ndef require_reader_controls(results, expected_count=4):\n    """Fail closed on missing, wrong, or capped control answers; no gold-based repair."""\n    if len(results) != expected_count or not all(\n        r.get(\'exact_match\') == 1 and not r.get(\'generation\', {}).get(\'hit_output_limit\', True)\n        for r in results\n    ):\n        raise RuntimeError(\'Reader control gate failed. Inspect saved answers before running benchmark questions.\')\n    return True\n'


## 3. GPU check, source checkout and immutable run settings

In [ ]:
import os, sys, subprocess, json, time, gc, hashlib, random, copy, importlib.util, ast, re, string
import numpy as np
import torch
import huggingface_hub
assert huggingface_hub.__version__ == '0.36.0', 'Run installation, restart the session, then resume at step 2.'
from huggingface_hub import HfApi
assert torch.cuda.is_available(), 'No GPU: select a GPU runtime, then rerun.'
# Exercise the same NF4/double-quantization path before downloading/loading an LLM.
import importlib.metadata
import bitsandbytes as bnb
BNB_EXPECTED = '0.50.2'
if bnb.__version__ != BNB_EXPECTED:
    raise RuntimeError(
        f'Loaded bitsandbytes {bnb.__version__}; expected {BNB_EXPECTED}. '
        'Update the install cell, run it, restart the session, then resume at step 2.')
if os.environ.get('BNB_CUDA_VERSION'):
    raise RuntimeError(
        'BNB_CUDA_VERSION is set. Remove this override and restart so bitsandbytes '
        'uses the CUDA version supplied by PyTorch.')
print('Python:', sys.version.split()[0], '| torch:', torch.__version__,
      '| torch CUDA:', torch.version.cuda, '| bitsandbytes:', bnb.__version__,
      '| GPU:', torch.cuda.get_device_name(0))

def check_nf4_cuda():
    layer = inputs = outputs = None
    try:
        layer = bnb.nn.Linear4bit(64, 64, bias=False, compute_dtype=torch.float16,
            compress_statistics=True, quant_type='nf4').half().to('cuda')
        inputs = torch.ones((2, 64), device='cuda', dtype=torch.float16)
        with torch.inference_mode():
            outputs = layer(inputs)
        torch.cuda.synchronize()
        assert torch.isfinite(outputs).all().item(), 'NF4 forward pass produced non-finite values.'
        print('NF4 CUDA quantization and forward-pass check passed.')
    except Exception as error:
        raise RuntimeError(
            'NF4 GPU compatibility check failed before model loading. '
            'Save the printed Python/torch/CUDA/bitsandbytes/GPU versions and this traceback. '
            'If you just upgraded bitsandbytes, restart the session first. '
            'This is not by itself evidence that the 4B model is too large.') from error
    finally:
        del layer, inputs, outputs
        gc.collect()
        torch.cuda.empty_cache()

check_nf4_cuda()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def save_json(path, obj):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    temp.replace(path)

def sha(text):
    return hashlib.sha256(text.encode()).hexdigest()

def checkout(url, dest, revision):
    dest = Path(dest)
    if not dest.exists():
        subprocess.run(['git', 'clone', '--quiet', url, str(dest)], check=True)
    subprocess.run(['git', '-C', str(dest), 'checkout', '--quiet', revision], check=True)
    actual = subprocess.check_output(['git', '-C', str(dest), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual == revision
    return dest

UPSTREAM = checkout('https://github.com/wangyu-ustc/Mem-alpha.git', '/content/Mem-alpha', MEMALPHA_COMMIT)
BENCHMARK = checkout('https://github.com/HUST-AI-HYZ/MemoryAgentBench.git', '/content/MemoryAgentBench', MAB_COMMIT)
sys.path.insert(0, str(UPSTREAM))
# Do not expose keys or environment variables. Fail if upstream unexpectedly tries a paid API.
import openai
def blocked_api(*args, **kwargs):
    raise RuntimeError('Paid API clients are disabled in this notebook.')
openai.OpenAI = blocked_api
openai.AzureOpenAI = blocked_api

settings = dict(writer=WRITER_ID, reader=READER_ID, embedding=EMBED_ID,
    chunk_tokens=CHUNK_TOKENS, write_tokens=WRITE_TOKENS, answer_tokens=ANSWER_TOKENS,
    max_input_plus_output=MAX_INPUT_PLUS_OUTPUT, questions=QUESTIONS_PER_CONTEXT,
    seed=SEED, memalpha_commit=MEMALPHA_COMMIT, benchmark_commit=MAB_COMMIT,
    quantization='NF4 double-quant float16 compute', including_core=True,
    writer_generation='greedy, thinking disabled', reader_mode='all stored memory',
    chunking='whole fact lines', notebook_protocol='week3-v2-reader-format', task=TASK, condition=CONDITION, reader_style=READER_STYLE, reader_thinking=READER_THINKING, split_sha256=sha(json.dumps(SPLIT, sort_keys=True)), protocol_sha256=sha(PROTOCOL_SOURCE), retrieval_k=6, parser='strict JSON before upstream execution')
manifest_path = OUTPUT / 'manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    assert manifest['settings'] == settings, 'Settings changed. Choose a new RUN_NAME.'
else:
    revisions = dict(PINNED_REVISIONS)
    assert all(m in revisions for m in [WRITER_ID, READER_ID, EMBED_ID])
    manifest = {'settings': settings, 'revisions': revisions, 'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    save_json(manifest_path, manifest)
revisions = manifest['revisions']
hardware = {'gpu': torch.cuda.get_device_name(0),
    'gpu_GiB': torch.cuda.get_device_properties(0).total_memory / 2**30,
    'torch': torch.__version__, 'python': sys.version, 'cuda': torch.version.cuda,
    'bitsandbytes': bnb.__version__, 'gpu_compute_capability': list(torch.cuda.get_device_capability(0)),
    'nf4_preflight': 'passed'}
save_json(OUTPUT / 'hardware.json', hardware)
(OUTPUT / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
print(json.dumps(hardware, indent=2))

exec(compile(PROTOCOL_SOURCE, 'week3_protocol', 'exec'))
save_json(OUTPUT / 'question_split.json', SPLIT)


## 4. Development-only data

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

ds = load_dataset('ai-hyz/MemoryAgentBench', revision=revisions['dataset'], split='Conflict_Resolution')
sources = [row['metadata']['source'] for row in ds]
print('Available sources:', sources)
targets = ['factconsolidation_sh_6k', 'factconsolidation_mh_6k']
selected = []
for source in targets:
    matches = [(i, row) for i, row in enumerate(ds) if row['metadata']['source'].lower() == source]
    assert len(matches) == 1, f'Expected one row for {source}; inspect the source list.'
    row_index, row = matches[0]
    assert len(row['questions']) == len(row['answers']) == 100
    assert sha(row['context']) == SPLIT['history_sha256']
    indices = SPLIT['tasks'][source]['development']
    assert len(indices) == 20 and not set(indices) & set(SPLIT['tasks'][source]['reserved_final'])
    assert all(isinstance(a, list) and a and all(isinstance(s, str) and s.strip() for s in a) for a in row['answers'])
    selected.append({'id': source, 'dataset_row': row_index, 'context': row['context'],
        'context_sha256': sha(row['context']), 'question_indices': indices,
        'questions': [row['questions'][i] for i in indices], 'answers': [row['answers'][i] for i in indices]})
save_json(OUTPUT / 'selected_examples.json', selected)
for item in selected:
    print(item['id'], 'characters:', len(item['context']), 'questions:', len(item['questions']))
    print('Context preview:', item['context'][:180])


## 5. Local CPU embeddings and memory tools

In [ ]:
from memory import Memory
from agent import MemoryAgent
from functions import get_memory_tool_schemas
import yaml
embedder = SentenceTransformer(EMBED_ID, revision=revisions[EMBED_ID], device='cpu')
embedding_stats = {'calls': 0, 'windows': 0}
class LocalMemory(Memory):
    def __init__(self):
        super().__init__(including_core=True)
        self.semantic_embedding_matrix = np.empty((0, 384), dtype=np.float32)
        self.episodic_embedding_matrix = np.empty((0, 384), dtype=np.float32)
    def _get_embedding(self, text):
        if not isinstance(text, str) or not text.strip():
            raise ValueError('Cannot embed an empty or non-text memory.')
        ids = embedder.tokenizer.encode(text, add_special_tokens=False)
        windows = [embedder.tokenizer.decode(ids[i:i+254]) for i in range(0, len(ids), 254)]
        vectors = embedder.encode(windows, normalize_embeddings=True, show_progress_bar=False)
        vector = np.mean(vectors, axis=0).astype(np.float32)
        norm = np.linalg.norm(vector)
        assert np.isfinite(vector).all() and norm > 0
        embedding_stats['calls'] += 1; embedding_stats['windows'] += len(windows)
        return vector / norm

# Prevent accidental use of the upstream paid embedding implementation anywhere else.
Memory._get_embedding = blocked_api

def snapshot(memory):
    return copy.deepcopy({'core': memory.core, 'semantic': memory.semantic, 'episodic': memory.episodic})

def restore(state):
    memory = LocalMemory()
    memory.core = state['core']
    for kind in ['semantic', 'episodic']:
        setattr(memory, kind, copy.deepcopy(state[kind]))
        pairs = [(k, v) for entry in state[kind] for k, v in entry.items()]
        setattr(memory, kind + '_embedding_ids', [k for k, v in pairs])
        matrix = np.stack([memory._get_embedding(v) for k, v in pairs]) if pairs else np.empty((0, 384), dtype=np.float32)
        setattr(memory, kind + '_embedding_matrix', matrix)
    return memory

m = LocalMemory()
m.new_memory_insert('semantic', 'Ada lives in Paris.')
mid = next(iter(m.semantic[0]))
m.memory_update('semantic', 'Ada lives in Rome.', memory_id=mid)
assert m.semantic[0][mid] == 'Ada lives in Rome.'
assert m.semantic_embedding_matrix.shape == (1, 384)
assert m.memory_search('semantic', 'Where does Ada live?', search_method='text-embedding')
m.memory_delete('semantic', memory_id=mid)
assert not m.semantic and m.semantic_embedding_matrix.shape == (0, 384)
assert snapshot(restore(snapshot(m))) == snapshot(m)
save_json(OUTPUT / 'tool_checks.json', {'insert_update_delete_restore': 'passed', 'paid_api_clients': 'blocked'})
print('CPU memory-tool checks passed.')
# Create parser/tool helper without loading a model or invoking its constructor.
helper = MemoryAgent.__new__(MemoryAgent)
WRITE_TEMPLATE = yaml.safe_load((UPSTREAM / 'config/prompts_wrt_datasource.yaml').read_text())['unified_prompt']


## 6. Load one model

In [ ]:
def load_lm(model_id):
    started = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revisions[model_id])
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revisions[model_id],
        quantization_config=quant, device_map={'': 0}, torch_dtype=torch.float16,
        attn_implementation='sdpa').eval()
    print('Loaded', model_id, 'in', round(time.perf_counter()-started, 1), 'seconds')
    return model, tokenizer

def generate_text(model, tokenizer, prompt, max_new_tokens):
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    input_length = inputs['input_ids'].shape[1]
    assert input_length + max_new_tokens <= MAX_INPUT_PLUS_OUTPUT, (
        f'Prompt {input_length} + response {max_new_tokens} exceeds configured limit; no truncation performed.')
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, temperature=1.0, top_p=1.0, top_k=50, max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    new_ids = output[0, input_length:]
    result = {'text': tokenizer.decode(new_ids, skip_special_tokens=True),
        'input_tokens': input_length, 'output_tokens': len(new_ids),
        'seconds': time.perf_counter()-started,
        'peak_allocated_GiB': torch.cuda.max_memory_allocated()/2**30,
        'peak_reserved_GiB': torch.cuda.max_memory_reserved()/2**30,
        'hit_output_limit': len(new_ids) >= max_new_tokens}
    del inputs, output, new_ids
    return result

writer, writer_tokenizer = load_lm(WRITER_ID)


## 7. Writer tests / memory construction
Strict JSON is required before execution. Failed tools roll back their individual mutations. A capped generation may still contain complete calls: those are executed and the cap remains flagged. A low rejection count alone is not a success criterion.

In [ ]:
def chunk_context(text, tokenizer, size):
    """Group whole lines (one fact per line) into chunks of at most ~size tokens.
    Week 1 cut every `size` tokens, which could split a fact across two chunks."""
    chunks, current, current_tokens = [], '', 0
    for line in text.splitlines(keepends=True):
        n = len(tokenizer(line, add_special_tokens=False)['input_ids'])
        if current and current_tokens + n > size:
            chunks.append(current)
            current, current_tokens = '', 0
        current += line
        current_tokens += n
    if current:
        chunks.append(current)
    assert ''.join(chunks) == text, 'Chunking altered the original text.'
    return chunks

def apply_write(memory, chunk, condition, model, tokenizer):
    started = time.perf_counter()
    before = snapshot(memory)
    retrieval_started = time.perf_counter()
    suffix, candidates = writer_suffix(before, chunk, condition)
    retrieval_seconds = time.perf_counter() - retrieval_started
    text = WRITE_TEMPLATE.format(context=chunk, max_new_tokens=int(WRITE_TOKENS*.8)) + suffix
    prompt = MemoryAgent.process_text_with_qwen_pipeline(text=text, tokenizer=tokenizer,
        functions=[t['function'] for t in get_memory_tool_schemas(memory)],
        status='memorie', enable_thinking=False, memory=memory, return_text=True)
    generation = generate_text(model, tokenizer, prompt, WRITE_TOKENS)
    accepted, rejected = strict_calls(helper._parse_response(generation['text']))
    calls = []
    for call in accepted:
        checkpoint = snapshot(memory)
        result = helper._run_tool_from_function_call(call, memory)
        error = ' error]' in result
        if error:
            memory = restore(checkpoint)  # rollback partially applied operations
        calls.append({'call':call, 'result':result, 'error':error})
    event = {'input_chunk':chunk, 'before':before, 'after':snapshot(memory),
        'rendered_prompt':prompt, 'retrieved_candidates':candidates,
        'generation':generation, 'calls':calls, 'rejected_calls':rejected,
        'raw_function_markers':generation['text'].count('✿FUNCTION✿'),
        'wall_seconds':time.perf_counter()-started, 'retrieval_seconds':retrieval_seconds, 'provenance':numbered_lines(snapshot(memory))}
    return memory, event
examples = selected
save_json(OUTPUT / 'all_examples.json', examples)
stage_started = time.perf_counter()
for example in examples:
    folder = OUTPUT / example['id']; folder.mkdir(exist_ok=True)
    chunks = chunk_context(example['context'], writer_tokenizer, CHUNK_TOKENS)
    save_json(folder / 'chunks.json', chunks)
    memory = LocalMemory()
    for index, chunk in enumerate(chunks):
        path = folder / f'chunk_{index:03d}.json'
        if path.exists():
            event = json.loads(path.read_text())
            assert event['input_chunk'] == chunk and event['before'] == snapshot(memory)
            memory = restore(event['after'])
            continue
        if time.perf_counter()-stage_started > MAX_STAGE_MINUTES*60:
            raise TimeoutError('Budget reached. Completed chunks saved; rerun this cell to resume.')
        memory, event = apply_write(memory, chunk, CONDITION, writer, writer_tokenizer)
        event.update(index=index, chunk_index=index, example_id=example['id'], condition=CONDITION)
        save_json(path, event)
        print(example['id'], index, 'capped', event['generation']['hit_output_limit'], 'rejected', len(event['rejected_calls']))
    save_json(folder / 'memory_final.json', snapshot(memory))


## 8. Release writer; load fixed reader

In [ ]:
if 'writer' in globals():
    del writer
if 'writer_tokenizer' in globals():
    del writer_tokenizer
gc.collect(); torch.cuda.empty_cache()
print('GPU allocated after release (GiB):', torch.cuda.memory_allocated()/2**30)
reader, reader_tokenizer = load_lm(READER_ID)


## 9. Reader controls and official scoring
For thinking runs, use a new run name and a sufficient answer budget; a missing final answer is scored as empty and flagged. Compare original versus explicit prompts on development only.

In [ ]:
# Import just the benchmark's pure scoring functions, avoiding its unrelated dependencies.
metric_names = {'normalize_answer', 'drqa_exact_match_score',
    'substring_exact_match_score', 'drqa_metric_max_over_ground_truths'}
metric_tree = ast.parse((BENCHMARK / 'utils/eval_other_utils.py').read_text())
metric_nodes = [n for n in metric_tree.body if isinstance(n, ast.FunctionDef) and n.name in metric_names]
assert {n.name for n in metric_nodes} == metric_names
metric_ns = {'re': re, 'string': string}
exec(compile(ast.Module(body=metric_nodes, type_ignores=[]), 'upstream_metrics', 'exec'), metric_ns)
def score(prediction, answers):
    return {name: float(metric_ns['drqa_metric_max_over_ground_truths'](metric_ns[fn], prediction, answers))
        for name, fn in [('exact_match', 'drqa_exact_match_score'), ('substring_exact_match', 'substring_exact_match_score')]}
assert score('The Rome.', ['Rome'])['exact_match'] == 1
assert score('Paris', ['Rome'])['substring_exact_match'] == 0

spec = importlib.util.spec_from_file_location('mab_templates', BENCHMARK / 'utils/templates.py')
templates = importlib.util.module_from_spec(spec); spec.loader.exec_module(templates)
query_template = templates.BASE_TEMPLATES['factconsolidation']['query']['long_context_agent']

def ask_reader(knowledge, question):
    messages = reader_messages(knowledge, question, READER_STYLE, query_template)
    prompt = reader_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=READER_THINKING)
    result = generate_text(reader, reader_tokenizer, prompt, ANSWER_TOKENS)
    result['rendered_prompt'] = prompt
    raw = result['text']
    result['raw_text'] = raw
    if READER_THINKING:
        result['text'] = raw.split('</think>', 1)[1].strip() if '</think>' in raw else ''
        result['missing_final_answer'] = '</think>' not in raw
    return result

controls = [
    ('1. Ada lives in Rome.', 'Where does Ada live?', ['Rome']),
    ('1. Ben likes chess.', 'What does Ben like?', ['chess']),
    ('1. Ada lives in Paris.\n2. Ada lives in Rome.', 'Where does Ada currently live?', ['Rome']),
    ('1. Mira works at Cedar Labs.\n2. Cedar Labs is in Oslo.', 'In which city does Mira work?', ['Oslo'])]
control_file = OUTPUT / 'reader_controls.json'
if control_file.exists():
    control_results = json.loads(control_file.read_text())
else:
    control_results = []
    for knowledge, question, answers in controls:
        result = ask_reader(knowledge, question)
        control_results.append({'question': question, 'answers': answers, 'generation': result, **score(result['text'], answers)})
    save_json(control_file, control_results)
print('Reader control exact matches:', sum(r['exact_match'] for r in control_results), '/', len(control_results))
if any(not r['exact_match'] for r in control_results):
    print('Inspect reader_controls.json. Reader errors may confound your memory experiment.')

READER_CONTROLS_PASSED = False
require_reader_controls(control_results, expected_count=4)
READER_CONTROLS_PASSED = True


In [ ]:
assert globals().get('READER_CONTROLS_PASSED', False), 'Run and pass the reader controls first.'
def memory_text(state):
    blocks = ['Core memory:\n' + (state['core'] or '')]
    for kind in ['semantic', 'episodic']:
        entries = [f'{key}: {value}' for entry in state[kind] for key, value in entry.items()]
        blocks.append(kind + ' memory:\n' + '\n'.join(entries))
    return '\n\n'.join(blocks)

stage_started = time.perf_counter()
for example in examples:
    folder = OUTPUT / example['id']
    state = json.loads((folder / 'memory_final.json').read_text())
    for condition, knowledge in [('stored_memory', memory_text(state)), ('full_context_control', example['context'])]:
        for qi, (question, answers) in enumerate(zip(example['questions'], example['answers'])):
            path = folder / f'answer_{condition}_{qi:03d}.json'
            if path.exists():
                continue
            if (time.perf_counter()-stage_started)/60 > MAX_STAGE_MINUTES:
                raise TimeoutError('Reader budget reached. Rerun this cell to resume saved questions.')
            result = ask_reader(knowledge, question)
            record = {'example_id': example['id'], 'condition': condition, 'question_index': example['question_indices'][qi], 'development_position': qi,
                'question': question, 'answers': answers, 'generation': result, **score(result['text'], answers)}
            save_json(path, record)
            print(example['id'], condition, qi, repr(result['text']), record['exact_match'])
print('Question answering complete.')


## Export
Upload this ZIP for results review. Completion means the selected workflow finished, not that the extension improved performance or that final evaluation is complete.

In [ ]:
assert globals().get('READER_CONTROLS_PASSED', False), 'Control checks have not passed.'
for example in examples:
    for qi in range(len(example['questions'])):
        assert (OUTPUT / example['id'] / f'answer_full_context_control_{qi:03d}.json').exists(), 'Benchmark answers are incomplete.'
        assert (OUTPUT / example['id'] / f'answer_stored_memory_{qi:03d}.json').exists(), 'Stored-memory answers are incomplete.'
save_json(OUTPUT / 'workflow_complete.json', {'task':TASK, 'gpu_execution_complete':True, 'manual_review_required':True, 'final_evaluation_run':False})
import shutil
archive = shutil.make_archive('/content/' + RUN_NAME, 'zip', root_dir=OUTPUT)
print('Saved in Drive:', OUTPUT)
from google.colab import files
files.download(archive)
